# Climate Comfort Index · Polynomial Ridge Regression

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts CCI +1, +2 and +3 days ahead*

This notebook trains and evaluates the **Polynomial Ridge Regression** model for the **Climate Comfort Index (CCI)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A regularised linear model with polynomial terms will **beat both naive baselines but fall short
of the tree ensembles**. CCI is a weighted sum of bounded scores, a largely additive structure a
linear model captures well, but the saturation of `rain_score` and `temp_score` at zero is a hard
threshold that polynomials approximate poorly and trees represent exactly.

Degree 2 tests whether pairwise interactions matter; the one with prior justification is
temperature with humidity (the same temperature feels worse in humid air).

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `StandardScaler` → `PolynomialFeatures` → `Ridge` | Interpretable and fast; sits between the baselines and the ensembles |
| Scaling | Before the expansion, fitted on training only | Unscaled products differ by orders of magnitude and break the ridge penalty |
| Degree | Tuned over 1 and 2 | Degree 3 on 105 features would create over 200,000 columns |
| Strategy | One model per horizon (the endpoint returns three values) | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 20 trials per target | See section 6 |
| Objective | validation **mean absolute error (MAE)**, in CCI points (lower is better) | Used for every comparison |

### Workflow

All ten experiment notebooks run the same code from `nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Final model against the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from nucleus_ml import CategoricalParameter, FloatParameter
from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "comfort_climate"
EXPERIMENT = "cci_polynomial_regression"
ALGORITHM = "Polynomial Ridge Regression"
N_TRIALS = 20  # Optuna trials per target

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It creates the experiment from a short `build_estimator` function, because this model's construction needs custom logic. It also sets a sensible untuned configuration (`default_params`), a cheaper configuration for the ablation (`ablation_params`) and the search space for section 6, written with the `nucleus_ml` parameter classes.

This model keeps a small `build_estimator` because it is a three-step pipeline: scaling happens
**before** the polynomial expansion so the products multiply comparable quantities, and the
expansion takes its own settings (`degree`, `interaction_only`). `interaction_only` keeps the
cross-products and drops the squares, roughly halving the column count. The ablation uses degree 1
to keep it fast.

In [ ]:
def build_estimator(params: dict, seed: int) -> Pipeline:
    """Scale, expand to polynomial terms, then fit a ridge regression."""
    params = dict(params)
    return Pipeline([
        ("scale", StandardScaler()),
        ("expand", PolynomialFeatures(degree=params.pop("degree"),
                                      interaction_only=params.pop("interaction_only"),
                                      include_bias=False)),
        ("ridge", Ridge(random_state=seed, **params)),
    ])


exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    build=build_estimator,
    default_params={"degree": 2, "interaction_only": True, "alpha": 10.0},
    ablation_params={"degree": 1, "interaction_only": False, "alpha": 10.0},
    search_space={
        "degree": CategoricalParameter([1, 2]),
        "interaction_only": CategoricalParameter([True, False]),
        "alpha": FloatParameter(1e-2, 1e4, log=True),
    },
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is
reserved for a single evaluation in the comparison notebook, once one model has been
selected, so nothing in this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up
> filter and an embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Two naive forecasts, both fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Persistence** | Tomorrow's CCI equals today's CCI | The current weather, but not the season |
| **Climatology** | The average CCI for that day of the year | The season, but not the current weather |

A model must beat **both** to be worth deploying: that shows it has combined current conditions
with the seasonal cycle.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging
what tuning bought in section 8; a search that fails to beat a
sensible default is a result worth reporting. `best_baseline` is repeated for
comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column), and the model is refitted at each step. This measures what each block of engineering contributes instead of assuming it. `ablation_params` are used throughout so the differences come from the features, not from tuning. Scores are on validation (MAE).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **mean absolute error (MAE)**, in CCI points (lower is better).

**Why MAE.** It is measured in CCI points, so an improvement of 0.4 reads directly as "0.4 comfort
points closer on average". RMSE squares errors and would be dominated by the rare heavy-rain days
where CCI collapses, tuning the model for the tail instead of the ordinary days that make up
almost all API traffic.

**Why one search per horizon.** Predictability decays with lead time, so a single averaged score
would let a strong *t*+1 model hide a weak *t*+3 one.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put future days into training and inflate every score, because consecutive days are strongly correlated. The fixed 2024 validation year matches how the model will be used: trained on the past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the
score moves by as much as the spread between the top search candidates
(`search_beats_noise = False`), the search was ranking noise and the conclusion
should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model

The selected configuration is refitted and compared with the untuned `default` configuration and every baseline (it appears as `tuned`). `gain_of_final_model` is positive when the final model is better than that row.

`pred_std` is shown next to the error metrics. A spread far below the observed spread means the
model is playing safe by predicting close to the average, which can give a respectable MAE while
missing exactly the uncomfortable days users care about. Expect error to grow with the horizon.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

This model has no built-in importance, so **permutation importance** is used: each feature is shuffled on the validation split and the drop in score is recorded. Features with negative importance are candidates for pruning.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature that looks unimportant on its own may belong to an essential block. The ablation in section 5 is the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model
generalises; a large gap means it has memorised patterns in the training years that
do not carry over to 2024.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to
`models/comfort_climate/experiments/cci_polynomial_regression/` with:

| File | Contents |
|:--|:--|
| `cci_polynomial_regression_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array
> at prediction time and would silently give wrong answers if the columns were
> permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **Did the expansion help?** Compare the best degree-1 and degree-2 trials in the search table.
  If degree 1 wins, the relationship is additive and the ensembles' advantage comes from the
  saturation thresholds rather than from interactions.
- **The selected penalty.** A very large `alpha` shrinks the model toward a constant, a direct
  statement about how little of the expansion is usable.
- **Against the ensembles.** A close result would be a genuine finding: this model is far cheaper
  to serve and far easier to explain to stakeholders.

> The test split has still not been used. It is scored exactly once, in the `cci_comparison` notebook, after one model has been selected on validation performance across all five CCI experiments.